# Stage-Two Static Feature Analysis

> **Safety boundary:** This workbook performs static analysis of sanitized excerpts, fake fixtures, and derived public metadata. It does not execute attacker code, inspect the analyst's environment, or contact any network.

> **Hostile-evidence warning:** This notebook derives features from a reviewed
> summary of hostile evidence. It does not execute the payload and does not
> publish the raw body.

## Publication-Safe Method

The private capture was inspected statically. The public derivative records
only its SHA-256, byte length, feature presence, and bounded behavior summary.
That supports reproducible claims without making the hostile body available as
an executable artifact.

In [ ]:
import json
from pathlib import Path

summary = json.loads(
    Path("../fixtures/stage2-feature-summary.public.json").read_text()
)
{
    "artifact": summary["artifact"],
    "sha256": summary["sha256"],
    "byte_length": summary["byte_length"],
    "raw_body_in_fixture": False,
}

In [ ]:
feature_rows = [
    {
        "feature": item["feature"],
        "present": "yes" if item["present"] else "not shown",
        "evidence_type": item["evidence_type"],
        "publication_form": item["publication_form"],
    }
    for item in summary["features"]
]
feature_rows

## Feature Interpretation

| Feature | Present | Evidence type | Publication form |
|---|---|---|---|
| host profiling | yes | static feature | summary only |
| OS, hostname, network interface | yes | static feature | summary only |
| process environment copy | yes | static feature | summary only |
| query-string beacon | yes | static feature | field names only |
| fixed polling interval | yes | static feature | 5000 ms |
| direct-IP C2 | yes | public IOC | published |
| response JSON parsing | yes | static feature | summary only |
| conditional remote task evaluation | yes | static feature | described, not implemented |
| wallet-file reads | not shown | not evidenced in captured stage | not claimed |
| credential-file enumeration | not shown | not evidenced in captured stage | not claimed |

In [ ]:
counts = {
    "present_features": sum(item["present"] for item in summary["features"]),
    "not_shown_features": sum(not item["present"] for item in summary["features"]),
    "network_features": sum(
        any(term in item["feature"] for term in ("query", "polling", "C2", "HTTP"))
        for item in summary["features"]
    ),
}
counts

## Capability Versus Captured Behavior

Conditional task evaluation makes arbitrary follow-on behavior possible if an
operator response is received. It does not prove that such a response arrived.
Likewise, the architecture could support later credential or wallet theft, but
those actions are not present in the captured stage and are not claimed.

**Observed:** downloaded stage-two JavaScript began executing and attempted the
published direct-IP connection.

**Static capability:** host profiling, environment copying, periodic beaconing,
and conditional operator task execution.

**Not proven:** successful direct-IP tasking or later file theft.

**Detection:** scan JavaScript for clusters of host profiling, environment
copying, query serialization, polling timers, direct-IP URLs, and response
content crossing into an execution primitive.

**Confidence upgrade:** a complete PCAP, proxy transcript, remote server logs,
or local telemetry showing a task response and its effects.